In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
from scipy.optimize import least_squares
from scipy.special import spherical_jn as jn
import emcee
import os
from multiprocessing import Pool
import corner

import utils.lib.nn_studio as nn_studio
import utils.lib.chiral_potential as chiral_potential
import utils.lib.granada_phases as granada
import utils.lib.auxiliary as aux
import utils.lib.lec_values as lec_values
from utils.lib.constants import *

import time
import datetime

In [2]:
%matplotlib widget
plt.close('all')

# Inferenza bayesiana NLO

Possiamo disaccoppiare i parametri su cui facciamo inferenza grazie alle simmetrie dell'interazione nucleare, in particolare avremo un blocco 2D e uno 3D.

Possiamo fare la separazione grazie alle seguenti regole di conservazione della Hamiltoniaa nucleare:
- spin totale $S$
- isospin totale $T$
- momento angolare totale $\vec{J} = \vec{L} + \vec{S}$
- parità $\pi = (-1)^L$

inoltre, trattandosi di un sistema di due fermioni identici, il principio di Pauli impone che la funzione d'onda totale sia completamente antisimmetrica, richiedendo che $L+S+T$ sia un numero dispari. 

In [3]:
T_Lab = granada.Tlabs 

shift_1S0 = granada.delta_1S0
shift_1S0_error = granada.delta_1S0_errors

shift_3S1 = granada.delta_3S1
shift_3S1_error = granada.delta_3S1_errors

shift_3D1 = granada.delta_3D1
shift_3D1_error = granada.delta_3D1_errors

shift_mix = granada.delta_3E1
shift_mix_error = granada.delta_3E1_errors

In [4]:
nn = nn_studio.nn_studio(jmin=0, jmax=1, tzmin=0, tzmax=0, Np=30, mesh_type="gauleg_infinite")
nn.Tlabs = T_Lab
nn.V = chiral_potential.two_nucleon_potential("NLO",Lambda=500.0)

_, channel_1S0 = nn.lookup_channel_idx(l=0,ll=0,s=0,j=0) 
_, channel_3S1 = nn.lookup_channel_idx(l=0,ll=2,s=1,j=1) 

## Blocco $^1S_0$

Consideriamo lo stato in onda S di singoletto: $L=0$ e $S=0$, che implica $J=0$. Per il principio di Pauli, affinché $L+S+T$ sia dispari ($0+0+T$), l'isospin deve essere necessariamente $T=1$. Lo stato quantistico completo è quindi:
$$| \psi \rangle = | J=0 , L=0 , S=0 \rangle | T=1 \rangle$$
L'Hamiltoniana nucleare potrebbe mescolare stati con diverso $L$, a patto di conservare $\pi$, $S$, $J$ e $T$. Tuttavia, non esiste alcun altro valore di momento angolare orbitale $L$ che, combinato con spin $S=0$, possa dare $J=0$. Di conseguenza, gli elementi di matrice di transizione verso altri stati sono rigorosamente nulli:
$$\langle J=0 , L=0 , S=0 | \mathcal{H} | \text{qualsiasi altro stato} \rangle = 0$$
L'onda $^1S_0$ è un canale isolato. Le osservabili ad essa associate dipendono esclusivamente dai parametri di contatto e di correzione ai momenti specifici per questo stato, ovvero $C_{^1S_0}$ e $D_{^1S_0}$

In [ ]:
# Inferenza su due soli parametri: C_1S0 e C_3S1.
# Gli altri LEC NLO sono fissati ai valori nominali.
def ln_prior_NLO(parameters):
    if len(parameters) != 2:
        return -np.inf

    C_1S0, C_3S1 = parameters

    avg_C1S0 = 0.0
    avg_C3S1 = 0.0
    sigma_C1S0 = 5.0
    sigma_C3S1 = 5.0

    ln_C1S0 = -0.5 * (np.log(2 * np.pi * sigma_C1S0**2) + ((C_1S0 - avg_C1S0) / sigma_C1S0) ** 2)
    ln_C3S1 = -0.5 * (np.log(2 * np.pi * sigma_C3S1**2) + ((C_3S1 - avg_C3S1) / sigma_C3S1) ** 2)

    # se un parametro e fuori di 3 sigma, il prior e -inf
    if abs(C_1S0 - avg_C1S0) > 3 * sigma_C1S0:
        return -np.inf
    if abs(C_3S1 - avg_C3S1) > 3 * sigma_C3S1:
        return -np.inf

    return ln_C1S0 + ln_C3S1


def ln_likelihood_NLO(parameters, nnstudio, chn_1S0, exp_1S0, err_1S0):
    NLO_lecs = {}
    NLO_lecs["C_1S0"] = parameters[0]
    NLO_lecs["C_3S1"] = parameters[1]

    # Parametri non inferiti: fissati ai valori nominali NLO
    NLO_lecs["D_1S0"] = lec_values.nlo_lecs["D_1S0"]
    NLO_lecs["D_3S1"] = lec_values.nlo_lecs["D_3S1"]
    NLO_lecs["D_3S1-3D1"] = lec_values.nlo_lecs["D_3S1-3D1"]
    NLO_lecs["D_1P1"] = lec_values.nlo_lecs["D_1P1"]
    NLO_lecs["D_3P0"] = lec_values.nlo_lecs["D_3P0"]
    NLO_lecs["D_3P1"] = lec_values.nlo_lecs["D_3P1"]
    NLO_lecs["D_3P2"] = lec_values.nlo_lecs["D_3P2"]

    nnstudio.lecs = NLO_lecs

    nnstudio.compute_Tmtx(chn_1S0, verbose=False)
    teo_1S0 = nnstudio.phase_shifts[0]

    ln_L_1S0 = -0.5 * np.sum(np.log(2 * np.pi * err_1S0**2) + ((exp_1S0 - teo_1S0) / err_1S0) ** 2)
    return ln_L_1S0


def ln_posterior_NLO(parameters, nnstudio, chn_1S0, exp_1S0, err_1S0):
    ln_pr = ln_prior_NLO(parameters)
    if not np.isfinite(ln_pr):
        return -np.inf

    ln_post = ln_pr
    ln_post += ln_likelihood_NLO(parameters, nnstudio, chn_1S0, exp_1S0, err_1S0)
    return ln_post


In [ ]:
# MCMC
ndim = 2
nwalkers = 32
nsteps = 1500

rng = np.random.default_rng(42)  # riproducibilita
initial_guess = np.array([lec_values.nlo_lecs["C_1S0"], lec_values.nlo_lecs["C_3S1"]])
initial_pos = initial_guess + 0.05 * rng.standard_normal((nwalkers, ndim))

# Evita oversubscription: oltre ~8 processi spesso il guadagno e marginale
ncpu = min(os.cpu_count() or 1, 8)

if __name__ == "__main__":
    with Pool(processes=ncpu) as pool:
        sampler = emcee.EnsembleSampler(
            nwalkers,
            ndim,
            ln_posterior_NLO,
            args=(nn, channel_1S0, shift_1S0, shift_1S0_error),
            pool=pool,
        )
        print(f"Eseguendo su {ncpu} core...")
        sampler.run_mcmc(initial_pos, nsteps, progress=True)

    print("\nFinito! :)\n")

## Blocco $^3S_1 - ^3D_1$

Consideriamo ora lo stato del deuterio, caratterizzato da spin $S=1$ e momento angolare totale $J=1$. Affinché la somma $L+S+T$ sia dispari, l'isospin deve essere $T=0$.Quali valori di $L$ possono generare $J=1$ accoppiandosi con $S=1$?$L=0$ (onda S, $\pi = +1$)$L=1$ (onda P, $\pi = -1$)$L=2$ (onda D, $\pi = +1$)Poiché l'interazione forte conserva la parità, lo stato in onda P non può mescolarsi con gli altri. Rimangono quindi le onde S e D, che hanno la stessa parità. L'operatore tensoriale $S_{12}$ presente nell'Hamiltoniana nucleare ha elementi di matrice non nulli tra questi due stati, generando un mescolamento (mixing):
$$\langle J=1 , L=0 , S=1  | \mathcal{H} | J=0 , L=2 , S=0  \rangle \neq 0$$
Il sistema non è più descritto da un singolo ket, ma da una matrice $2 \times 2$ nello spazio $\left\{ | ^3S_1 \rangle, | ^3D_1 \rangle \right\}$. Lo scattering e le proprietà dello stato legato (come il momento di quadrupolo) in questo blocco dipendono esclusivamente e congiuntamente dalle costanti $C_{^3S_1}$, $D_{^3S_1}$ e dal termine di accoppiamento tensoriale $D_{^3E_1}$.